# 지금까지 실험한 모델 한눈에 보기

이 노트북은 모델을 처음 보는 학습자를 위한 **최종 성적표**입니다. 상세한 실험 과정은
`07_baseline_comparison.ipynb`와 `08_embedding_comparison.ipynb`에 두고, 여기서는
지금까지 실험한 22개 모델을 같은 조건에서 비교합니다.

## 공통 평가 조건

- 데이터: RFP 10개, 요구사항 1,024건
- 동결 앵커 100건: 라벨 생성 때 이미 예시로 사용됐으므로 검증·평가에서 제외
- 실제 평가 합계: 비앵커 요구사항 924건
- 분할: 문서 단위 LODO 10겹, 각 fold는 학습 8 / 검증 1 / 평가 1문서
- 주 지표: 세 클래스를 같은 무게로 보는 macro F1

```text
요구사항 원문 → 숫자 표현(TF-IDF 또는 E5) → 분류기(Logistic 또는 SVC) → 라벨
```

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display
from matplotlib import font_manager

ROOT = next((p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
             if (p / 'scripts').is_dir()), Path.cwd().resolve())
sys.path.insert(0, str(ROOT))
installed_fonts = {font.name for font in font_manager.fontManager.ttflist}
plt.rcParams['font.family'] = next(
    font for font in ['Malgun Gothic', 'NanumGothic', 'Noto Sans CJK KR', 'DejaVu Sans']
    if font in installed_fonts
)
plt.rcParams['axes.unicode_minus'] = False

from scripts.evaluation.baselines import (
    CHAR_BALANCED, CHAR_LENGTH_BALANCED, CHAR_NUMBERS_BALANCED,
    CHAR_STRUCTURE_BALANCED, CHAR_TYPE_BALANCED, CHAR_UNWEIGHTED, DUMMY,
    ELASTIC_NET_STRUCTURE, SVD_STRUCTURE_LOGISTIC, SVD_STRUCTURE_XGBOOST,
    SVM_BALANCED, WORD_BALANCED, WORD_CHAR_BALANCED,
    WORD_CHAR_COMPLEMENT_NB, WORD_CHAR_TYPE_BALANCED,
    run_lodo, run_review_weight_tuned_lodo, run_type_weight_tuned_lodo, summarize,
)
from scripts.evaluation.embeddings import (
    E5_LOGISTIC, E5_SVM, TFIDF_E5, TFIDF_E5_NUMBERS,
    load_cached_embeddings, run_embedding_lodo, run_hybrid_lodo,
)
from scripts.labeling.label_dataset import load_label_dataset

rows, meta = load_label_dataset()
cache_path = ROOT / 'data/processed/multilingual-e5-small.npz'
embeddings = load_cached_embeddings(cache_path, rows)
if embeddings is None:
    raise FileNotFoundError(
        'README의 embeddings 명령으로 현재 환경에 맞는 임베딩 캐시를 먼저 만드세요'
    )
print(f"데이터셋 {meta['dataset_version']} / 전체 {len(rows)}건 / 평가 924건")

In [ ]:
runs = {
    DUMMY.name: run_lodo(rows, DUMMY),
    WORD_BALANCED.name: run_lodo(rows, WORD_BALANCED),
    CHAR_UNWEIGHTED.name: run_lodo(rows, CHAR_UNWEIGHTED),
    CHAR_BALANCED.name: run_lodo(rows, CHAR_BALANCED),
    SVM_BALANCED.name: run_lodo(rows, SVM_BALANCED),
    WORD_CHAR_BALANCED.name: run_lodo(rows, WORD_CHAR_BALANCED),
    CHAR_TYPE_BALANCED.name: run_lodo(rows, CHAR_TYPE_BALANCED),
    WORD_CHAR_TYPE_BALANCED.name: run_lodo(rows, WORD_CHAR_TYPE_BALANCED),
    'char + 유형 검증 weight': run_type_weight_tuned_lodo(rows, CHAR_TYPE_BALANCED),
    'word+char + 유형 검증 weight': run_type_weight_tuned_lodo(rows, WORD_CHAR_TYPE_BALANCED),
    CHAR_LENGTH_BALANCED.name: run_lodo(rows, CHAR_LENGTH_BALANCED),
    CHAR_NUMBERS_BALANCED.name: run_lodo(rows, CHAR_NUMBERS_BALANCED),
    CHAR_STRUCTURE_BALANCED.name: run_lodo(rows, CHAR_STRUCTURE_BALANCED),
    ELASTIC_NET_STRUCTURE.name: run_lodo(rows, ELASTIC_NET_STRUCTURE),
    SVD_STRUCTURE_LOGISTIC.name: run_lodo(rows, SVD_STRUCTURE_LOGISTIC),
    SVD_STRUCTURE_XGBOOST.name: run_lodo(rows, SVD_STRUCTURE_XGBOOST),
    WORD_CHAR_COMPLEMENT_NB.name: run_lodo(rows, WORD_CHAR_COMPLEMENT_NB),
    'LinearSVC + 검증 weight': run_review_weight_tuned_lodo(rows, SVM_BALANCED),
    E5_LOGISTIC.name: run_embedding_lodo(rows, embeddings, E5_LOGISTIC),
    E5_SVM.name: run_embedding_lodo(rows, embeddings, E5_SVM),
    TFIDF_E5.name: run_hybrid_lodo(rows, embeddings, TFIDF_E5),
    TFIDF_E5_NUMBERS.name: run_hybrid_lodo(rows, embeddings, TFIDF_E5_NUMBERS),
}
assert all(sum(result.test_size for result in run) == 924 for run in runs.values())

scores = pd.DataFrame([{
    '모델': name,
    'macro F1': summarize(run)['macro_f1']['fold_mean'],
    '정확도': summarize(run)['accuracy']['fold_mean'],
    '계약 precision': summarize(run)['review_precision']['fold_mean'],
    '계약 recall': summarize(run)['review_recall']['fold_mean'],
    '계약 F1': summarize(run)['review_f1']['fold_mean'],
} for name, run in runs.items()]).set_index('모델').sort_values('macro F1', ascending=False)
display(scores.style.format('{:.3f}').highlight_max(
    subset=['macro F1', '정확도', '계약 recall'], color='#b7e4c7'
))
print(f"평균 점수 1위: {scores.index[0]} / macro F1 {scores.iloc[0]['macro F1']:.3f}")

In [ ]:
plot_data = scores.sort_values('macro F1')
fig, axes = plt.subplots(1, 3, figsize=(16, 6), sharey=True)
for ax, metric, color in zip(
    axes, ['macro F1', '정확도', '계약 recall'], ['#2a9d8f', '#457b9d', '#e76f51']
):
    ax.barh(plot_data.index, plot_data[metric], color=color)
    ax.set_title(metric)
    ax.set_xlim(0, 0.75)
    ax.grid(axis='x', alpha=0.25)
    for i, value in enumerate(plot_data[metric]):
        ax.text(value + 0.01, i, f'{value:.3f}', va='center', fontsize=9)
fig.suptitle('같은 8/1/1 LODO에서 모델 22종 비교', fontsize=15)
plt.tight_layout()
plt.show()

In [ ]:
winner = runs[CHAR_BALANCED.name]
comparisons = []
for name in [WORD_CHAR_BALANCED.name, CHAR_TYPE_BALANCED.name,
             WORD_CHAR_TYPE_BALANCED.name, 'char + 유형 검증 weight',
             'word+char + 유형 검증 weight', WORD_CHAR_COMPLEMENT_NB.name,
             CHAR_LENGTH_BALANCED.name, CHAR_NUMBERS_BALANCED.name,
             CHAR_STRUCTURE_BALANCED.name, ELASTIC_NET_STRUCTURE.name,
             SVD_STRUCTURE_LOGISTIC.name, SVD_STRUCTURE_XGBOOST.name,
             SVM_BALANCED.name, E5_LOGISTIC.name, E5_SVM.name,
             TFIDF_E5.name, TFIDF_E5_NUMBERS.name]:
    deltas = [after.macro_f1 - before.macro_f1
              for before, after in zip(winner, runs[name])]
    comparisons.append({
        'TF-IDF Logistic과 비교': name,
        '평균 차이': sum(deltas) / len(deltas),
        '최소': min(deltas),
        '최대': max(deltas),
        '이긴 fold': f'{sum(delta > 0 for delta in deltas)}/10',
    })
fold_comparison = pd.DataFrame(comparisons).set_index('TF-IDF Logistic과 비교')
display(fold_comparison.style.format({
    '평균 차이': '{:+.3f}', '최소': '{:+.3f}', '최대': '{:+.3f}',
}))
print('평균만 우연히 높아진 것이 아니라 여러 문서에서 반복해서 이기는지도 함께 봅니다.')

## 최종 해석

평균만 보면 문자 TF-IDF+E5 결합이 macro F1 **0.611**로 가장 높습니다. 하지만
기존 문자 Logistic(0.601)보다 차이가 +0.010이고, fold별로는 -0.029~+0.045이며
우세도 5/10이라 안정적 개선으로 확정하지 않습니다. 따라서 현재 권장 파이프라인은 여전히
**요구사항 원문 → 문자 3~4gram TF-IDF → balanced Logistic Regression**입니다.

- 문자+단어 Logistic: 0.603이지만 문서별 방향이 불규칙해 잡음입니다.
- ComplementNB: macro F1 0.505, 계약 recall 0.351로 명확히 낮았습니다.
- LinearSVC: 같은 TF-IDF 입력에서도 macro F1이 0.579로 낮았습니다.
- 검토 클래스 추가 가중치: recall을 조금 회복했지만 전체 macro F1은 개선하지 못했습니다.
- 동결 E5 임베딩 단독: Logistic 0.544, LinearSVC 0.546으로 TF-IDF보다 낮았습니다.
- 문자 TF-IDF+E5: 0.611로 숫자상 최고지만 우세 5/10이라 유망 후보로 둡니다.
- 문자 TF-IDF+E5+숫자: 0.594로 숫자 없는 결합보다 0.017 낮았습니다.
- 문자+유형: 0.549로 문자 단독보다 -0.051, 우세 2/10이었습니다.
- 단어+문자+유형: 0.558로 단어+문자보다 -0.046, 우세 1/10이었습니다.
- 유형 가중치 검증 선택: 각각 0.589와 0.584로 회복했지만 본문 기준선보다 낮았습니다.
- 구조·수치 결합: 글자 수 0.571, 숫자 정보 0.591, 전체 결합 0.572였습니다.
- Elastic-net 구조·수치 결합: 0.471로 모든 평가 문서에서 낮았습니다.
- SVD100+Logistic/XGBoost: 각각 0.528/0.564로 원래 희소 TF-IDF보다 낮았습니다.

따라서 지금은 **복잡한 모델보다 정확한 위험 표현을 직접 포착하는 문자 TF-IDF가 더
효과적**입니다. 이것은 파인튜닝까지 나쁘다는 뜻이 아니라, 범용 임베딩을 고정한 채
입력으로 쓰는 방법이 현재 데이터에서는 기준선을 넘지 못했다는 뜻입니다.